# KOSIS 2차 READY 실행 (뉴스 5,000건 실험)

`hcx_early_bge_extracted.csv`에서 1차 게이트를 다시 계산한 뒤 다음 과정을 한 번에 실행합니다.

```text
HCX measurement → 1차 mapping_eligible → measurement 기반 BGE/Reranker
→ KOSIS 공식 메타 → ITEM/OBJ 조합 검증 → 2차 mapping_status=READY
→ READY 실제값 조회 및 verdict
```

위에서부터 순서대로 실행합니다. BGE 검색은 GPU를 사용하지만 KOSIS 메타/API 단계는 GPU를 거의 사용하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import subprocess
import sys

subprocess.run(['nvidia-smi'], check=True)

REPO_URL = 'https://github.com/rnwjdgus03/NLP_05-Team-Project-3.git'
BRANCH = 'codex/repro-baseline-20260727'
REPO_DIR = Path('/content/NLP_05-Team-Project-3')
DRIVE_ROOT = Path('/content/drive/MyDrive/NLP_05-Team-Project-3')
INDEX_DIR = DRIVE_ROOT / 'indexes' / 'kosis_bge_m3'
SOURCE_RUN_DIR = DRIVE_ROOT / 'runs' / 'early_bge_rag_5000'
HCX_INPUT = SOURCE_RUN_DIR / 'hcx_early_bge_extracted.csv'
RUN_DIR = DRIVE_ROOT / 'runs' / 'second_ready_5000'

RUN_DIR.mkdir(parents=True, exist_ok=True)
print('입력:', HCX_INPUT)
print('결과:', RUN_DIR)

## 1. 코드와 패키지 준비

In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH,
        REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=1.26,<3',
    'sentence-transformers>=3.4,<6',
    'transformers>=4.45,<6',
    'requests>=2.31,<3',
    'python-dotenv>=1.0,<2',
], check=True)
print('코드 준비 완료:', REPO_DIR)

## 2. HCX 결과 확인

Drive에서 자동으로 찾습니다. 파일이 없을 때만 업로드 창이 열립니다.

In [ ]:
from google.colab import files
import pandas as pd

if not HCX_INPUT.exists():
    print('hcx_early_bge_extracted.csv를 선택하세요.')
    uploaded = files.upload()
    expected = 'hcx_early_bge_extracted.csv'
    if expected not in uploaded:
        raise RuntimeError(f'{expected} 파일을 선택하세요.')
    SOURCE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    HCX_INPUT.write_bytes(uploaded[expected])

hcx = pd.read_csv(HCX_INPUT, encoding='utf-8-sig')
required_columns = {
    'claim_id', 'claim_measurement_id', 'claim_text',
    'measurement_indicator', 'value', 'unit',
    'measurement_period', 'measurement_prd_se',
}
missing_columns = sorted(required_columns - set(hcx.columns))
if missing_columns:
    raise RuntimeError(f'HCX 결과 필수 컬럼 누락: {missing_columns}')

manifest = INDEX_DIR / 'manifest.json'
if not manifest.exists():
    raise FileNotFoundError(f'BGE 인덱스가 없습니다: {manifest}')

print('HCX 행:', len(hcx))
print('HCX claim:', hcx['claim_id'].nunique())
print('입력과 BGE 인덱스: 준비 완료')

## 3. KOSIS API 키 확인

Colab 왼쪽 열쇠 아이콘의 보안 비밀에 `KOSIS_API_KEY`를 등록합니다.

In [ ]:
from google.colab import userdata

if not os.environ.get('KOSIS_API_KEY'):
    os.environ['KOSIS_API_KEY'] = userdata.get('KOSIS_API_KEY') or ''
if not os.environ.get('KOSIS_API_KEY'):
    raise RuntimeError('Colab 보안 비밀에 KOSIS_API_KEY를 등록하세요.')
print('KOSIS API 키: 준비 완료')

## 4. 1차 게이트부터 2차 READY와 실제값 검증까지 실행

BGE 검색 이후 KOSIS 메타와 실제 API를 순차 호출하므로 시간이 오래 걸릴 수 있습니다.

In [ ]:
TABLE_INDEX = REPO_DIR / 'data/reference/kosis_table_summary.csv'
if not TABLE_INDEX.exists():
    raise FileNotFoundError(f'KOSIS 통계표 인덱스가 없습니다: {TABLE_INDEX}')

command = [
    sys.executable, '-u', str(REPO_DIR / 'run_kosis_measurement_pipeline.py'),
    '--input', str(HCX_INPUT),
    '--table-index', str(TABLE_INDEX),
    '--out-dir', str(RUN_DIR),
    '--top-tables', '5',
    '--top-rank-for-meta', '2',
    '--top-meta', '8',
    '--min-score', '10',
    '--retrieval-mode', 'hybrid',
    '--semantic-index', str(INDEX_DIR),
    '--semantic-top-k', '50',
    '--rerank-top-k', '20',
    '--device', 'cuda',
    '--item-top-k', '3',
    '--obj-top-k', '2',
    '--max-combinations', '20',
    '--delay', '0.10',
    '--verify',
]
subprocess.run(command, check=True)

## 5. 2차 READY 결과 요약과 전용 CSV 생성

In [ ]:
stem = HCX_INPUT.stem
FIRST_READY_CSV = RUN_DIR / f'{stem}_kosis_ready.csv'
REJECTED_CSV = RUN_DIR / f'{stem}_kosis_rejected.csv'
TABLE_CANDIDATES_CSV = RUN_DIR / f'{stem}_kosis_table_candidates.csv'
META_CSV = RUN_DIR / f'{stem}_kosis_meta_index.csv'
CANDIDATES_WITH_META_CSV = RUN_DIR / f'{stem}_kosis_candidates_with_meta.csv'
VALIDATED_CSV = RUN_DIR / f'{stem}_kosis_validated_mappings.csv'
VERIFIED_CSV = RUN_DIR / f'{stem}_kosis_verified.csv'
SECOND_READY_CSV = RUN_DIR / f'{stem}_mapping_ready.csv'

validated = pd.read_csv(VALIDATED_CSV, encoding='utf-8-sig')
second_ready = validated[validated['mapping_status'] == 'READY'].copy()
second_ready.to_csv(SECOND_READY_CSV, index=False, encoding='utf-8-sig')

first_ready = pd.read_csv(FIRST_READY_CSV, encoding='utf-8-sig')
rejected = pd.read_csv(REJECTED_CSV, encoding='utf-8-sig')
verified = pd.read_csv(VERIFIED_CSV, encoding='utf-8-sig')

measurement_key = 'claim_measurement_id'
print('1차 mapping_eligible measurement:', first_ready[measurement_key].nunique())
print('2차 mapping_status=READY 행:', len(second_ready))
print('2차 READY measurement:', second_ready[measurement_key].nunique())
print('2차 READY claim:', second_ready['claim_id'].nunique())
print('\nmapping_status 분포')
display(validated['mapping_status'].value_counts(dropna=False))
print('\nmapping_reason 상위')
display(validated['mapping_reason'].value_counts(dropna=False).head(20))

verdict_column = 'verdict' if 'verdict' in verified.columns else 'verification_result'
if verdict_column in verified.columns:
    print('\n최종 verdict 분포')
    display(verified[verdict_column].value_counts(dropna=False))

print('\n2차 READY 파일:', SECOND_READY_CSV)
print('검증 전체 파일:', VALIDATED_CSV)
print('실제값 결과 파일:', VERIFIED_CSV)